In [81]:
import pandas as pd
import numpy as np

df = pd.read_csv("files/Synthetic_Demand_Forecasting_Dataset_50K/Fact_Demand_Raw_50K.csv")

df.shape
df.head()
df.info()
df.describe(include="all")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50100 entries, 0 to 50099
Data columns (total 47 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Date                    50100 non-null  object 
 1   Product_ID              50100 non-null  object 
 2   Region                  50100 non-null  object 
 3   Category                50100 non-null  object 
 4   Brand                   50100 non-null  object 
 5   Product_Tier            50100 non-null  object 
 6   Channel                 50100 non-null  object 
 7   Customer_Segment        50100 non-null  object 
 8   Demand_Units            50100 non-null  int64  
 9   Units_Sold              50100 non-null  int64  
 10  Lost_Sales_Units        50100 non-null  int64  
 11  List_Price              50100 non-null  float64
 12  Selling_Price           50000 non-null  float64
 13  Discount_Pct            50100 non-null  float64
 14  Promotion_Flag          50100 non-null

,Date,Product_ID,Region,Category,Brand,Product_Tier,Channel,Customer_Segment,Demand_Units,Units_Sold,...,Inflation_Pct,Interest_Rate_Pct,GDP_Growth_Pct,Consumer_Confidence,Fuel_Price,Supplier_Reliability,Supplier_Delay_Flag,Campaign_ID,Holiday_Flag,Festival_Flag
count,50100,50100,50100,50100,50100,50100,50100,50100,50100.000000,50100.000000,...,50100.000000,50100.000000,50100.000000,50100.000000,50100.000000,50100.000000,50100.000000,50100.000000,50100.000000,50100.000000
unique,250,25,17,5,12,3,12,6,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,2023-03-20,P025,Pune,Electronics,Veloxa,Standard,Retail Store,Mass Market,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,203,2008,6145,10026,11106,22044,16144,12022,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,172.204731,170.529122,...,5.227270,5.626853,6.962419,100.586359,96.590998,0.875222,0.124411,2.162515,0.091996,0.103932
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,166.805941,163.209265,...,0.520024,0.901177,0.242123,3.993686,4.938488,0.047692,0.330053,6.245637,0.289023,0.305176
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.000000,4.000000,...,4.320000,3.960000,6.270000,92.240000,84.620000,0.750000,0.000000,0.000000,0.000000,0.000000
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,52.000000,52.000000,...,4.810000,4.590000,6.860000,97.090000,93.130000,0.835000,0.000000,0.000000,0.000000,0.000000
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,111.000000,110.000000,...,5.070000,6.200000,6.980000,101.830000,98.100000,0.869000,0.000000,0.000000,0.000000,0.000000
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,244.000000,242.000000,...,5.720000,6.340000,7.110000,103.750000,100.600000,0.916000,0.000000,0.000000,0.000000,0.000000


In [82]:
df.isnull().sum().sort_values(ascending=False)

Selling_Price             100
Marketing_Spend           100
Competitor_Price          100
Temperature_C             100
Humidity_Pct              100
Rainfall_mm                 0
Return_Units                0
Impressions                 0
Clicks                      0
Conversions                 0
Store_Traffic               0
Search_Index                0
Date                        0
Lead_Time_Days              0
Inflation_Pct               0
Interest_Rate_Pct           0
GDP_Growth_Pct              0
Consumer_Confidence         0
Fuel_Price                  0
Supplier_Reliability        0
Supplier_Delay_Flag         0
Campaign_ID                 0
Holiday_Flag                0
Stockout_Flag               0
Closing_Inventory           0
Reorder_Point               0
Product_ID                  0
Region                      0
Category                    0
Brand                       0
Product_Tier                0
Channel                     0
Customer_Segment            0
Demand_Uni

In [83]:
#Selling Price=List Price×(1−(Discount_Pct)/100)
df["Selling_Price"] = df["Selling_Price"].fillna(
    df["List_Price"] * (1 - df["Discount_Pct"] / 100)  
)

In [84]:
#Competitor Price=List Price×Competitor Price Index
df["Competitor_Price"] = df["Competitor_Price"].fillna(
    df["List_Price"] * df["Competitor_Price_Index"]
)

In [85]:
#Since weather is common to a region + week, copy the value from another row with the same Region and Date.
df["Temperature_C"] = df["Temperature_C"].fillna(
    df.groupby(["Region", "Date"])["Temperature_C"].transform("first")
)

In [86]:
df["Humidity_Pct"] = df["Humidity_Pct"].fillna(
    df.groupby(["Region", "Date"])["Humidity_Pct"].transform("first")
)

In [87]:
#using median
df["Marketing_Spend"] = df["Marketing_Spend"].fillna(
    df.groupby(["Product_ID", "Region"])["Marketing_Spend"]
      .transform("median")
)

In [88]:
df[[
    "Selling_Price",
    "Competitor_Price",
    "Temperature_C",
    "Humidity_Pct",
    "Marketing_Spend"
]].isnull().sum()

Selling_Price       0
Competitor_Price    0
Temperature_C       0
Humidity_Pct        0
Marketing_Spend     0
dtype: int64

In [89]:
df.isnull().sum().sum()

np.int64(0)

checking for duplicates

In [90]:
df = df.drop_duplicates()

In [91]:
df.shape

(50000, 47)

fixing categorical inconsistencies

In [92]:
df["Brand"] = df["Brand"].str.strip().str.title()

df["Channel"] = df["Channel"].str.strip().str.title()

df["Region"] = df["Region"].str.strip().str.title()

df["Region"] = df["Region"].replace({
    "Bangalore": "Bengaluru"
})

In [93]:
df["Channel"].value_counts()
df["Region"].value_counts()
df["Brand"].value_counts()

Brand
Solaris    12000
Veloxa     12000
Kairo      12000
Zenora      6000
Nimbus      4000
Trident     4000
Name: count, dtype: int64

In [94]:
dim_region = pd.read_csv("files/Synthetic_Demand_Forecasting_Dataset_50K/Dim_Region.csv")

In [95]:
canon = {
    r.lower(): r
    for r in dim_region["Region"].dropna().unique()
}

In [96]:
canon["bangalore"] = "Bengaluru"

In [97]:
df["Region"] = (
    df["Region"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(canon)
)

In [98]:
df["Region"].value_counts(dropna=False)

Region
Pune         6250
Ahmedabad    6250
Kolkata      6250
Chennai      6250
Bengaluru    6250
Mumbai       6250
Hyderabad    6250
Delhi NCR    6250
Name: count, dtype: int64

In [99]:
for col in ["Brand", "Category",
            "Channel", "Promotion_Flag"]:

    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
    )

In [100]:
df["Selling_Price"] = df["Selling_Price"].fillna(
    df["List_Price"] *
    (1 - df["Discount_Pct"] / 100)
).round(2)

In [101]:
df["Competitor_Price"] = df["Competitor_Price"].fillna(
    df["List_Price"] *
    df["Competitor_Price_Index"]
).round(2)

In [102]:
df["Temperature_C"] = df["Temperature_C"].fillna(
    df.groupby(["Region", "Date"])["Temperature_C"]
      .transform("first")
)

In [103]:
df["Humidity_Pct"] = df["Humidity_Pct"].fillna(
    df.groupby(["Region", "Date"])["Humidity_Pct"]
      .transform("first")
)

Fixing invalid discounts

In [104]:
invalid_discount = ~df["Discount_Pct"].between(0, 60)

print("Invalid discounts:", invalid_discount.sum())

Invalid discounts: 60


In [105]:
df.loc[invalid_discount, "Discount_Pct"] = (
    (
        df.loc[invalid_discount, "List_Price"]
        - df.loc[invalid_discount, "Selling_Price"]
    )
    / df.loc[invalid_discount, "List_Price"]
    * 100
).round(2)

In [106]:
df["Discount_Pct"] = df["Discount_Pct"].clip(0, 60).round(2)

In [107]:
df["Discount_Pct"].describe()

count    50000.000000
mean         5.414984
std          5.885283
min          0.010000
25%          2.250000
50%          3.390000
75%          5.010000
max         29.660000
Name: Discount_Pct, dtype: float64

In [113]:
df = (
    df.sort_values(
        ["Product_ID", "Region", "Date"]
    )
    .reset_index(drop=True)
)

checking business logic

In [114]:
calculated_revenue = (
    df["Units_Sold"] *
    df["Selling_Price"]
)

revenue_error = (
    df["Revenue"] - calculated_revenue
).abs()

print("Maximum revenue difference:", revenue_error.max())
print("Within tolerance:", (revenue_error < 1).all())

Maximum revenue difference: 4.656612873077393e-10
Within tolerance: True


In [115]:
calculated_margin = (
    df["Units_Sold"] *
    (df["Selling_Price"] - df["Unit_Cost"])
)

margin_error = (
    df["Gross_Margin"] - calculated_margin
).abs()

print("Maximum margin difference:", margin_error.max())
print("Within tolerance:", (margin_error < 1).all())

Maximum margin difference: 4.656612873077393e-10
Within tolerance: True


In [116]:
#standout
pd.crosstab(
    df["Stockout_Flag"],
    df["Lost_Sales_Units"] > 0
)

Lost_Sales_Units,False,True
Stockout_Flag,,
0,48470,0
1,0,1530


In [118]:
df["Date"] = pd.to_datetime(df["Date"])

In [119]:
assert df.shape[0] == 50000
assert df.isnull().sum().sum() == 0

assert df[
    ["Date", "Product_ID", "Region"]
].duplicated().sum() == 0

assert df["Region"].nunique() == 8

assert pd.api.types.is_datetime64_any_dtype(df["Date"])

assert (df["Selling_Price"] <= df["List_Price"]).all()

In [120]:
print("All data validation checks passed.")

All data validation checks passed.


In [121]:
df.to_csv(
    "files/Synthetic_Demand_Forecasting_Dataset_50K/Fact_Demand_Clean_50K.csv",
    index=False
)

In [122]:
db_df = df.copy()

db_df.columns = (
    db_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)